# EMIASD 25/26 - Projet de Qualité de Données : Capture de la provenance

**Enseignant :** Khalid Belhajjame  
**Étudiant :**
 Issam Yousr


## 1 - Objectifs du projet

L’objectif principal de ce projet est de développer un système de capture de provenance efficace pour les données utilisées dans les systèmes d’information.

La provenance des données permet de conserver une trace de l’origine des données après différentes transformations. Elle est essentielle pour garantir la qualité, la traçabilité et la fiabilité des données utilisées dans les processus décisionnels.

Plus précisément, ce projet vise à répondre aux questions suivantes :

- D’où vient une ligne de sortie après une ou plusieurs transformations ?
- Quelles lignes d’entrée ont contribué à produire une ligne finale ?
- Quelles lignes de sortie dépendent d’une ligne d’entrée donnée ?
- De quelles colonnes d’origine proviennent les colonnes finales après transformation ?
- Comment conserver cette information de provenance de manière efficace en mémoire ?



## 2 - Démarche proposée

Nous commencerons par définir la classe `RowProv`, pour capturer la provenance des données sur les opérations impactant **les lignes d'un DataFrame**. Cette classe sera conçue pour suivre les transformations de données à travers différentes opérations telles que :
- Le filtrage (`filter` ainsi que `dropna` car utilisé dans les pipelines de nettoyage) ;
- La jointure (`merge`) ;
- L'union (`concat`).

Nous profiterons de cette conception pour évaluer les performances en termes d’espace de stockage nécessaire pour stocker les tenseurs, ainsi que leur complexité de traitement requis pour les requêtes de provenance.

Nous définirons également la classe `ColumnProv` pour capturer la provenance des données au niveau des colonnes d'un DataFrame. Cette classe suivra les transformations de données à travers des opérations telles que :
- La sélection de colonnes `select`;
- La transformation de colonnes `transform`;
- L'agrégation de colonnes `aggregate` ;
- L'augmentation de données `oversampling`.


Nous testerons ensuite ces classes sur un jeu de données jouet afin de valider leur bon fonctionnement.

Nous nous assurerons que les classes `RowProv` et `ColumnProv` sont capables de capturer la provenance de manière précise et efficace, en vérifiant que les tenseurs de provenance reflètent correctement les transformations appliquées aux données sur les pipelines proposés suivants :
- CensusCleanup ;
- CompassCleanup ;
- GermanCleanup.


## 3 - Imports et fonctions utilitaires


In [281]:
import os
import time
import sys
import numpy as np
import pandas as pd
import random
from scipy import sparse
from scipy.sparse import issparse

try:
    from IPython.display import display
except Exception:
    display = print


Pour ce projet, il est nécessaire d'implémenter des fonctions addtionnelles pour faciliter l’exploitation des matrices de provenance. Nous avons ainsi défini deux types les deux fonctions suivantes :

- `output_to_inputs` : retrouver les lignes d’entrée qui ont contribué à une ligne de sortie donnée ;
- `input_to_outputs` : retrouver les lignes de sortie qui dépendent d’une ligne d’entrée donnée.

Ces requêtes permettent d’analyser la provenance dans les deux sens :

- du résultat final vers les données sources ;
- des données sources vers les résultats produits.

In [282]:
def output_to_inputs(mapping, out_row):
    """Donne les lignes d'entrée qui ont contribué à une ligne de sortie."""
    if isinstance(mapping, list):
        return [output_to_inputs(m, out_row) for m in mapping]
    m = mapping.tocsc()
    return list(m[:, out_row].nonzero()[0])


def input_to_outputs(mapping, input_row):
    """Donne les lignes de sortie qui dépendent d'une ligne d'entrée."""
    if isinstance(mapping, list):
        return [input_to_outputs(m, input_row) for m in mapping]
    m = mapping.tocsr()
    return list(m[input_row, :].nonzero()[1])

def highlight(text):
    """Met en gras dans la sortie console."""
    return f"[1m{text}[0m"



Afin d'évaluer la performance des fonctions de capture de provenance, nous avons également défini des fonctions de benchmark pour mesurer le temps d'exécution et la mémoire utilisée :
- `tensor_nbytes`: calculer la mémoire utilisée par un tenseur de provenance, en fonction de son format (dense ou creux) et de sa taille.
- `dense_theoretical_nbytes`: calcule la taille théorique d'une matrice dense équivalente, afin de comparer avec la mémoire utilisée par la matrice creuse et évaluer le gain mémoire obtenu ;
- `mapping_summary`: afficher un résumé de la matrice de provenance, notamment le nombre de lignes d’entrée et de sortie, ainsi que le nombre de connexions (valeurs à 1) dans la matrice.


In [283]:


def tensor_nbytes(matrix):
    """Retourne la taille mémoire réelle d'une matrice sparse scipy."""
    if sparse.issparse(matrix):
        return matrix.data.nbytes + matrix.indices.nbytes + matrix.indptr.nbytes if hasattr(matrix, "indptr") else matrix.data.nbytes + matrix.row.nbytes + matrix.col.nbytes
    return sys.getsizeof(matrix)


def dense_theoretical_nbytes(shape, dtype=np.int8):
    """Taille théorique si la matrice était stockée de façon dense."""
    return int(np.prod(shape) * np.dtype(dtype).itemsize)

def mapping_summary(mapping, name="tensor"):
    """Résumé rapide d'un tenseur de provenance."""
    if isinstance(mapping, list):
        rows = []
        for i, m in enumerate(mapping):
            rows.append({
                "nom": f"{name}_{i+1}",
                "shape": m.shape,
                "nnz": m.nnz,
                "taille_sparse_octets": tensor_nbytes(m),
                "taille_dense_theorique_octets": dense_theoretical_nbytes(m.shape),
                "dense_vs_sparse": round(dense_theoretical_nbytes(m.shape) / max(tensor_nbytes(m), 1), 2),
            })
        return pd.DataFrame(rows)
    return pd.DataFrame([{
        "nom": name,
        "shape": mapping.shape,
        "nnz": mapping.nnz,
        "taille_sparse_octets": tensor_nbytes(mapping),
        "taille_dense_theorique_octets": dense_theoretical_nbytes(mapping.shape),
        "dense_vs_sparse": round(dense_theoretical_nbytes(mapping.shape) / max(tensor_nbytes(mapping), 1), 2),
    }])

## 4 - Implémentation de la classe `RowProv`

La classe `RowProv` permet de capturer la provenance au niveau des lignes d’un DataFrame.

Pour chaque opération, on construit une matrice binaire creuse de taille :
$$
n_{in} \times n_{out}
$$
où, lorsque : 
$$
M[i, j] = 1
$$
signifie que la ligne `i` du DataFrame d’entrée contribue à la ligne `j` du DataFrame de sortie.

Cette représentation permet de suivre précisément les liens entre les lignes avant et après transformation, tout en limitant l’espace mémoire grâce à l’utilisation de matrices creuses.

La classe `RowProv` est utilisée comme un décorateur ou wrapper autour des opérations de transformation. Elle permet de retourner à la fois :

- le DataFrame transformé ;
- la matrice de provenance associée.

Les opérations supportées sont notamment :

- `filter` : sélection de lignes selon une condition ;
- `dropna` : suppression des lignes contenant des valeurs manquantes ;
- `merge` : jointure entre deux DataFrames ;
- `concat` : concaténation verticale de plusieurs DataFrames ;
- `oversampling` : duplication ou augmentation de certaines lignes, notamment dans le cas de classes déséquilibrées.

In [284]:
class RowProv:
    """
    Décorateur de provenance pour les opérations qui modifient les lignes.

    Cette classe applique le pattern décorateur :
    - elle reçoit une fonction de transformation ;
    - elle intercepte l'appel avec __call__ ;
    - elle exécute l'opération ;
    - elle valide le résultat ;
    - elle enregistre des métadonnées sur l'opération ;
    - elle retourne le DataFrame transformé et le tenseur de provenance.
    """

    SUPPORTED_OPERATIONS = {"filter", "dropna", "merge", "concat", "oversample"}

    def __init__(self, func, operation=None):
        self.func = func
        self.operation = operation or func.__name__
        self.history = []
        self.last_metadata = None

    def __call__(self, *args, **kwargs):
        """
        Intercepte l'appel à la fonction décorée.

        Le décorateur ajoute un comportement autour de l'opération :
        - validation de l'opération ;
        - mesure du temps d'exécution ;
        - contrôle du format retourné ;
        - stockage de métadonnées de provenance.
        """

        if self.operation not in self.SUPPORTED_OPERATIONS:
            raise NotImplementedError(
                f"La capture de provenance pour l'opération "
                f"'{self.operation}' n'est pas encore implémentée."
            )

        start_time = time.perf_counter()

        # Exécution de la fonction décorée
        result = self.func(*args, **kwargs)

        execution_time = time.perf_counter() - start_time

        # Validation du format retourné
        if not isinstance(result, tuple) or len(result) != 2:
            raise ValueError(
                f"L'opération '{self.operation}' doit retourner un tuple "
                f"(dataframe_resultat, tenseur_provenance)."
            )

        output_df, provenance_tensor = result

        # Validation du DataFrame de sortie
        if not isinstance(output_df, pd.DataFrame):
            raise TypeError(
                f"L'opération '{self.operation}' doit retourner un DataFrame "
                f"comme premier élément."
            )

        # Validation du ou des tenseurs de provenance
        if isinstance(provenance_tensor, list):
            for tensor in provenance_tensor:
                if not issparse(tensor):
                    raise TypeError(
                        f"Chaque tenseur de provenance de l'opération "
                        f"'{self.operation}' doit être une matrice sparse."
                    )
        else:
            if not issparse(provenance_tensor):
                raise TypeError(
                    f"Le tenseur de provenance de l'opération "
                    f"'{self.operation}' doit être une matrice sparse."
                )

        # Création des métadonnées de provenance
        metadata = {
            "operation": self.operation,
            "function": self.func.__name__,
            "output_shape": output_df.shape,
            "execution_time_sec": execution_time,
        }

        if isinstance(provenance_tensor, list):
            metadata["tensor_shapes"] = [tensor.shape for tensor in provenance_tensor]
            metadata["tensor_nnz"] = [tensor.nnz for tensor in provenance_tensor]
        else:
            metadata["tensor_shape"] = provenance_tensor.shape
            metadata["tensor_nnz"] = provenance_tensor.nnz
            metadata["tensor_density"] = (
                provenance_tensor.nnz
                / (provenance_tensor.shape[0] * provenance_tensor.shape[1])
            )

        self.last_metadata = metadata
        self.history.append(metadata)

        return output_df, provenance_tensor
    

### 4.1 Implémentation du Filtrage / réduction horizontale


In [285]:
from scipy.sparse import coo_matrix

def row_filter(df, condition):
    """Filtrage par condition pandas, ex. 'age > 30'."""
    mask = df.eval(condition).fillna(False).astype(bool)
    out = df.loc[mask].copy().reset_index(drop=True)

    input_rows = np.flatnonzero(mask.to_numpy())
    output_rows = np.arange(len(input_rows))
    data = np.ones(len(input_rows), dtype=np.int8)

    tensor = sparse.csr_matrix(
        (data, (input_rows, output_rows)),
        shape=(len(df), len(out)),
        dtype=np.int8,
    )
    return out, tensor


### 4.2 Implémentation du `dropna` / suppression de lignes


In [286]:
def row_dropna(df, **kwargs):
    """Capture la provenance d'un dropna, y compris si subset/how sont fournis."""
    out = df.dropna(**kwargs)

    input_positions = df.index.get_indexer(out.index)
    output_positions = np.arange(len(out))
    data = np.ones(len(out), dtype=np.int8)

    tensor = sparse.csr_matrix(
        (data, (input_positions, output_positions)),
        shape=(len(df), len(out)),
        dtype=np.int8,
    )
    return out.reset_index(drop=True), tensor


### 4.3 Implémentation de la Jointure / fusion de données


In [287]:
def row_merge(left, right, **kwargs):
    """Capture la provenance d'une jointure pandas.merge.

    Fonctionne pour inner, left, right et outer join. Les lignes non appariées
    d'un côté ne créent pas de contribution dans le tenseur correspondant.
    """
    left_tmp = left.copy().reset_index(drop=True)
    right_tmp = right.copy().reset_index(drop=True)

    left_marker = "__prov_left_row__"
    right_marker = "__prov_right_row__"
    left_tmp[left_marker] = np.arange(len(left_tmp))
    right_tmp[right_marker] = np.arange(len(right_tmp))

    out = pd.merge(left_tmp, right_tmp, **kwargs)
    n_out = len(out)

    left_raw = out[left_marker].to_numpy()
    right_raw = out[right_marker].to_numpy()

    left_valid = ~pd.isna(left_raw)
    right_valid = ~pd.isna(right_raw)

    left_rows = left_raw[left_valid].astype(int)
    left_cols = np.arange(n_out)[left_valid]
    right_rows = right_raw[right_valid].astype(int)
    right_cols = np.arange(n_out)[right_valid]

    left_tensor = sparse.csr_matrix(
        (np.ones(len(left_rows), dtype=np.int8), (left_rows, left_cols)),
        shape=(len(left), n_out),
        dtype=np.int8,
    )
    right_tensor = sparse.csr_matrix(
        (np.ones(len(right_rows), dtype=np.int8), (right_rows, right_cols)),
        shape=(len(right), n_out),
        dtype=np.int8,
    )

    out = out.drop(columns=[left_marker, right_marker])
    return out.reset_index(drop=True), [left_tensor, right_tensor]


### 4.4 Implémentation du  Union / concaténation


In [288]:
def row_concat(df_list, **kwargs):
    """Capture la provenance d'une concaténation verticale.

    Le résultat contient une matrice par DataFrame d'entrée.
    """
    dfs = [df.copy().reset_index(drop=True) for df in df_list]
    out = pd.concat(dfs, axis=0, ignore_index=True, sort=False, **kwargs)

    tensors = []
    col_offset = 0
    out_len = len(out)

    for df in dfs:
        n = len(df)
        rows = np.arange(n)
        cols = np.arange(col_offset, col_offset + n)
        tensor = sparse.csr_matrix(
            (np.ones(n, dtype=np.int8), (rows, cols)),
            shape=(n, out_len),
            dtype=np.int8,
        )
        tensors.append(tensor)
        col_offset += n

    return out, tensors


### 4.5 Implémentation de l'Oversampling / augmentation horizontale

L'oversampling ajoute des lignes. Dans ce projet, on capture la provenance d'un oversampling simple par duplication de lignes minoritaires. Les nouvelles lignes dupliquées gardent donc la même origine que la ligne source.


In [289]:
def random_oversample(df, target_col, random_state=42):
    """Fonction simple d'oversampling aléatoire par duplication."""
    return row_random_oversample(df, target_col=target_col, random_state=random_state)[0]


def row_random_oversample(df, target_col, random_state=42):
    rng = np.random.default_rng(random_state)
    df_reset = df.copy().reset_index(drop=True)

    counts = df_reset[target_col].value_counts()
    max_count = counts.max()

    selected_indices = list(range(len(df_reset)))

    for cls, count in counts.items():
        if count < max_count:
            class_indices = df_reset.index[df_reset[target_col] == cls].to_numpy()
            extra = rng.choice(class_indices, size=max_count - count, replace=True)
            selected_indices.extend(extra.tolist())

    out = df_reset.iloc[selected_indices].reset_index(drop=True)

    input_rows = np.array(selected_indices, dtype=int)
    output_rows = np.arange(len(selected_indices))
    tensor = sparse.csr_matrix(
        (np.ones(len(selected_indices), dtype=np.int8), (input_rows, output_rows)),
        shape=(len(df_reset), len(out)),
        dtype=np.int8,
    )
    return out, tensor


## 5 - Implémentation de la classe `ColumnProv`

La classe `ColumnProv` permet de suivre l’origine des colonnes après différentes transformations du schéma du DataFrame.

Elle est utile lorsque les opérations ne modifient pas forcément le nombre de lignes, mais transforment les colonnes.

Elle permet par exemple de suivre :

- la sélection de colonnes ;
- la suppression de colonnes ;
- le renommage de colonnes ;
- la création de nouvelles colonnes ;
- l’encodage de variables catégorielles ;
- le one-hot encoding, où une colonne d’origine peut produire plusieurs colonnes finales.

Ainsi, la provenance colonne permet de savoir de quelles variables initiales dépendent les variables finales.

In [290]:
class ColumnProv:
    """Provenance simple des colonnes sous forme de dictionnaire colonne_finale -> colonnes_sources."""

    def __init__(self, df):
        self.df = df.copy()
        self.final_columns = {col: {col} for col in df.columns}

    def select(self, cols):
        self.final_columns = {col: self.final_columns[col] for col in cols if col in self.final_columns}

    def drop_columns(self, cols):
        for col in cols:
            self.final_columns.pop(col, None)

    def rename_column(self, from_col, to_col):
        if from_col in self.final_columns:
            self.final_columns[to_col] = self.final_columns.pop(from_col)

    def add_column(self, new_col, from_cols=None):
        self.final_columns[new_col] = set(from_cols or [])

    def transform_column(self, col, from_cols=None):
        if col in self.final_columns:
            self.final_columns[col] = set(from_cols or [col])

    def to_dummies(self, cols, df_before_dummies=None):
        source_df = self.df if df_before_dummies is None else df_before_dummies.copy()

        for col in cols:
            if col not in self.final_columns:
                continue
            self.final_columns.pop(col, None)
            values = pd.Series(source_df[col].dropna().unique()).sort_values().tolist()
            for val in values:
                self.final_columns[f"{col}_{val}"] = {col}

    def to_dataframe(self):
        return pd.DataFrame({
            "Colonne finale": list(self.final_columns.keys()),
            "Colonne(s) d'origine": [", ".join(sorted(origins)) if origins else "Aucune" for origins in self.final_columns.values()],
        })

    def print(self, col=None):
        if col is not None:
            origins = self.final_columns.get(col, set())
            print(f"La colonne {col!r} provient de : {', '.join(sorted(origins)) if origins else 'Aucune'}")
        display(self.to_dataframe())


## 6 - Tests de la classe `RowProv` sur un jeu de données jouet


Préparation des données de test :

Nous créons un ensemble de données jouet contenant des étudiants et leurs cours pour valider le bon fonctionnement de la classe `RowProv`.

In [291]:
df_student = pd.DataFrame({
    "id": [10, 20, 30, 40, 50, 60, 70, 80],
    "age": [18, 22, 31, 40, 38, None, 25, 30],
    "sex": ["M", "F", "M", "F", "M", "F", "M", "F"],
    "score": [85, 90, 78, 92, 88, 95, 80, None],
})

df_courses = pd.DataFrame({
    "id": [10, 20, 20, 30, 50, 50, 70],
    "course": ["Maths", "Physique", "Biologie", "Maths", "Histoire", "Maths", "Physique"],
})

display(df_student)
display(df_courses)


,id,age,sex,score
0,10,18.0,M,85.0
1,20,22.0,F,90.0
2,30,31.0,M,78.0
3,40,40.0,F,92.0
4,50,38.0,M,88.0
5,60,NaN,F,95.0
6,70,25.0,M,80.0
7,80,30.0,F,NaN


,id,course
0,10,Maths
1,20,Physique
2,20,Biologie
3,30,Maths
4,50,Histoire
5,50,Maths
6,70,Physique


### 6.1 Test du filtre


In [292]:
filter_with_prov = RowProv(row_filter, operation="filter")
filtered_df, filter_tensor = filter_with_prov(df_student, "age > 30")

display(filtered_df)
display(mapping_summary(filter_tensor, "filter_tensor"))

for out_row in range(len(filtered_df)):
    print(f"La ligne de sortie {out_row} vient de la ligne d'entrée {output_to_inputs(filter_tensor, out_row)}")


,id,age,sex,score
0,30,31.0,M,78.0
1,40,40.0,F,92.0
2,50,38.0,M,88.0


,nom,shape,nnz,taille_sparse_octets,taille_dense_theorique_octets,dense_vs_sparse
0,filter_tensor,"(8, 3)",3,51,24,0.47


La ligne de sortie 0 vient de la ligne d'entrée [np.int32(2)]
La ligne de sortie 1 vient de la ligne d'entrée [np.int32(3)]
La ligne de sortie 2 vient de la ligne d'entrée [np.int32(4)]


In [293]:
filter_with_prov.last_metadata

{'operation': 'filter',
 'function': 'row_filter',
 'output_shape': (3, 4),
 'execution_time_sec': 0.000650875001156237,
 'tensor_shape': (8, 3),
 'tensor_nnz': 3,
 'tensor_density': 0.125}

### 6.2. Test de `dropna`


In [294]:
dropna_with_prov = RowProv(row_dropna, operation="dropna")
dropna_df, dropna_tensor = dropna_with_prov(df_student)

display(dropna_df)
display(mapping_summary(dropna_tensor, "dropna_tensor"))

for out_row in range(len(dropna_df)):
    print(f"La ligne de sortie {out_row} vient de la ligne d'entrée {output_to_inputs(dropna_tensor, out_row)}")


,id,age,sex,score
0,10,18.0,M,85.0
1,20,22.0,F,90.0
2,30,31.0,M,78.0
3,40,40.0,F,92.0
4,50,38.0,M,88.0
5,70,25.0,M,80.0


,nom,shape,nnz,taille_sparse_octets,taille_dense_theorique_octets,dense_vs_sparse
0,dropna_tensor,"(8, 6)",6,66,48,0.73


La ligne de sortie 0 vient de la ligne d'entrée [np.int32(0)]
La ligne de sortie 1 vient de la ligne d'entrée [np.int32(1)]
La ligne de sortie 2 vient de la ligne d'entrée [np.int32(2)]
La ligne de sortie 3 vient de la ligne d'entrée [np.int32(3)]
La ligne de sortie 4 vient de la ligne d'entrée [np.int32(4)]
La ligne de sortie 5 vient de la ligne d'entrée [np.int32(6)]


### 6.3. Test du merge


In [295]:
merge_with_prov = RowProv(row_merge, operation="merge")
merged_df, merge_tensors = merge_with_prov(df_student, df_courses, on="id", how="inner")

display(merged_df)
display(mapping_summary(merge_tensors, "merge_tensor"))

for out_row in range(len(merged_df)):
    left_origin, right_origin = output_to_inputs(merge_tensors, out_row)
    print(f"Ligne finale {out_row} <- étudiant {left_origin} + cours {right_origin}")


,id,age,sex,score,course
0,10,18.0,M,85.0,Maths
1,20,22.0,F,90.0,Physique
2,20,22.0,F,90.0,Biologie
3,30,31.0,M,78.0,Maths
4,50,38.0,M,88.0,Histoire
5,50,38.0,M,88.0,Maths
6,70,25.0,M,80.0,Physique


,nom,shape,nnz,taille_sparse_octets,taille_dense_theorique_octets,dense_vs_sparse
0,merge_tensor_1,"(8, 7)",7,71,56,0.79
1,merge_tensor_2,"(7, 7)",7,67,49,0.73


Ligne finale 0 <- étudiant [np.int32(0)] + cours [np.int32(0)]
Ligne finale 1 <- étudiant [np.int32(1)] + cours [np.int32(1)]
Ligne finale 2 <- étudiant [np.int32(1)] + cours [np.int32(2)]
Ligne finale 3 <- étudiant [np.int32(2)] + cours [np.int32(3)]
Ligne finale 4 <- étudiant [np.int32(4)] + cours [np.int32(4)]
Ligne finale 5 <- étudiant [np.int32(4)] + cours [np.int32(5)]
Ligne finale 6 <- étudiant [np.int32(6)] + cours [np.int32(6)]


### 6.4. Test de concat / union


In [296]:
concat_with_prov = RowProv(row_concat, operation="concat")
concat_df, concat_tensors = concat_with_prov([filtered_df, df_student.head(2)])

display(concat_df)
display(mapping_summary(concat_tensors, "concat_tensor"))

for out_row in range(len(concat_df)):
    print(f"Ligne finale {out_row} <- origines {output_to_inputs(concat_tensors, out_row)}")


,id,age,sex,score
0,30,31.0,M,78.0
1,40,40.0,F,92.0
2,50,38.0,M,88.0
3,10,18.0,M,85.0
4,20,22.0,F,90.0


,nom,shape,nnz,taille_sparse_octets,taille_dense_theorique_octets,dense_vs_sparse
0,concat_tensor_1,"(3, 5)",3,31,15,0.48
1,concat_tensor_2,"(2, 5)",2,22,10,0.45


Ligne finale 0 <- origines [[np.int32(0)], []]
Ligne finale 1 <- origines [[np.int32(1)], []]
Ligne finale 2 <- origines [[np.int32(2)], []]
Ligne finale 3 <- origines [[], [np.int32(0)]]
Ligne finale 4 <- origines [[], [np.int32(1)]]


### 6.5. Test de l'oversampling


In [297]:
oversample_with_prov = RowProv(row_random_oversample, operation="oversample")

small_df = pd.DataFrame({
    "id": [1, 2, 3, 4, 5],
    "score": [10, 12, 11, 18, 19],
    "label": [0, 0, 0, 1, 1],
})

over_df, over_tensor = oversample_with_prov(
    small_df,
    target_col="label",
    random_state=0
)

display(small_df)
display(over_df)
display(mapping_summary(over_tensor, "oversampling_tensor"))

for out_row in range(len(over_df)):
    print(f"Ligne oversamplée {out_row} <- ligne source {output_to_inputs(over_tensor, out_row)}")

,id,score,label
0,1,10,0
1,2,12,0
2,3,11,0
3,4,18,1
4,5,19,1


,id,score,label
0,1,10,0
1,2,12,0
2,3,11,0
3,4,18,1
4,5,19,1
5,5,19,1


,nom,shape,nnz,taille_sparse_octets,taille_dense_theorique_octets,dense_vs_sparse
0,oversampling_tensor,"(5, 6)",6,54,30,0.56


Ligne oversamplée 0 <- ligne source [np.int32(0)]
Ligne oversamplée 1 <- ligne source [np.int32(1)]
Ligne oversamplée 2 <- ligne source [np.int32(2)]
Ligne oversamplée 3 <- ligne source [np.int32(3)]
Ligne oversamplée 4 <- ligne source [np.int32(4)]
Ligne oversamplée 5 <- ligne source [np.int32(4)]


## 7 - Tests de la classe `ColumnProv`


In [298]:
col_prov = ColumnProv(df_student)
col_prov.rename_column("sex", "gender")
col_prov.add_column("score_plus_10", from_cols=["score"])
col_prov.drop_columns(["score"])
col_prov.print()

col_prov_2 = ColumnProv(df_student)
col_prov_2.to_dummies(["sex"], df_before_dummies=df_student)
col_prov_2.print()


,Colonne finale,Colonne(s) d'origine
0,id,id
1,age,age
2,gender,sex
3,score_plus_10,score


,Colonne finale,Colonne(s) d'origine
0,id,id
1,age,age
2,score,score
3,sex_F,sex
4,sex_M,sex


Remarque : l'opération de filtrage est ici représentée par une fonction personnalisée `row_filter`.
On évite volontairement d'utiliser `pd.DataFrame.filter`, car cette méthode pandas filtre des colonnes ou des labels, et non des lignes selon une condition booléenne.
Le décorateur `RowProv` reste donc explicite : chaque opération est identifiée par un nom métier (`filter`, `dropna`, `merge`, `concat`, `oversample`).

## 8 - Test sur des transformations successives

Le projet ne se limite pas à des opérations isolées. Il montre aussi comment suivre la provenance dans un pipeline composé de plusieurs transformations successives.

Pour cela, les matrices de provenance sont composées entre elles afin d’obtenir une provenance globale entre le dataset initial et le dataset final.

Cela permet de conserver une traçabilité complète même lorsque plusieurs opérations sont enchaînées, par exemple :

- nettoyage des valeurs manquantes ;
- filtrage ;
- jointure ;
- concaténation ;
- oversampling.
- 
Dans cette section, on illustre une chaîne de traitement plus réaliste composée de trois opérations successives :

1. suppression des lignes contenant des valeurs manquantes ;
2. filtrage des individus ayant un score supérieur ou égal à 10 ;
3. sur-échantillonnage de la classe minoritaire.

L'objectif est de montrer que la provenance peut être propagée sur plusieurs étapes successives à l'aide d'une composition de matrices creuses.

In [299]:
# Exemple de pipeline avec trois opérations successives :
# 1. dropna
# 2. filter
# 3. oversampling

df_initial = pd.DataFrame({
    "id": [1, 2, 3, 4, 5, 6],
    "score": [8, 12, np.nan, 15, 7, 20],
    "label": [0, 0, 1, 1, 0, 1]
})

print("Dataset initial")
display(df_initial)

# Étape 1 : suppression des valeurs manquantes
df_step1, T1 = row_dropna(df_initial)

print("Après dropna")
display(df_step1)
display(mapping_summary(T1, "T1_dropna"))

# Étape 2 : filtrage des lignes avec score >= 10
df_step2, T2 = row_filter(df_step1, "score >= 10")

print("Après filtrage score >= 10")
display(df_step2)
display(mapping_summary(T2, "T2_filter"))

# Étape 3 : oversampling de la classe minoritaire
df_final, T3 = row_random_oversample(
    df_step2,
    target_col="label",
    random_state=0
)

print("Dataset final après oversampling")
display(df_final)
display(mapping_summary(T3, "T3_oversampling"))

Dataset initial


,id,score,label
0,1,8.0,0
1,2,12.0,0
2,3,NaN,1
3,4,15.0,1
4,5,7.0,0
5,6,20.0,1


Après dropna


,id,score,label
0,1,8.0,0
1,2,12.0,0
2,4,15.0,1
3,5,7.0,0
4,6,20.0,1


,nom,shape,nnz,taille_sparse_octets,taille_dense_theorique_octets,dense_vs_sparse
0,T1_dropna,"(6, 5)",5,53,30,0.57


...

,id,score,label
0,2,12.0,0
1,4,15.0,1
2,6,20.0,1


,nom,shape,nnz,taille_sparse_octets,taille_dense_theorique_octets,dense_vs_sparse
0,T2_filter,"(5, 3)",3,39,15,0.38


Dataset final après oversampling


,id,score,label
0,2,12.0,0
1,4,15.0,1
2,6,20.0,1
3,2,12.0,0


,nom,shape,nnz,taille_sparse_octets,taille_dense_theorique_octets,dense_vs_sparse
0,T3_oversampling,"(3, 4)",4,36,12,0.33


In [300]:
# Composition globale de la provenance :
# df_initial -> df_step1 -> df_step2 -> df_final
#
# Dans ce notebook, les tenseurs sont orientés comme suit :
# lignes d'entrée x lignes de sortie
#
# T1 : initial  x step1
# T2 : step1    x step2
# T3 : step2    x final
#
# Donc :
# T_global = T1 @ T2 @ T3
#
# T_global relie directement chaque ligne du dataset initial
# aux lignes du dataset final.

print("Formes des matrices :")
print("T1_dropna       :", T1.shape)
print("T2_filter       :", T2.shape)
print("T3_oversampling :", T3.shape)

T_global = T1 @ T2 @ T3
T_global = T_global.astype(bool).astype(np.int8).tocsr()

display(mapping_summary(T_global, "T_global_pipeline_3_operations"))

print("Provenance sortie finale -> entrée initiale")
for out_row in range(T_global.shape[1]):
    input_rows = output_to_inputs(T_global, out_row)
    if len(input_rows) > 0:
        print(f"Ligne finale {out_row} <- ligne(s) initiale(s) {input_rows[0]}")
    else:
        print(f"Ligne finale {out_row} <- aucune ligne initiale (exclue par le filtre)")

print("\nProvenance entrée initiale -> sortie finale")
for in_row in range(T_global.shape[0]):
    output_rows = input_to_outputs(T_global, in_row)
    if len(output_rows) > 0:
        print(f"Ligne initiale {in_row} -> ligne(s) finale(s) {output_rows[0]}")
    else:
        print(f"Ligne initiale {in_row} -> aucune ligne finale (exclue par le filtre)")

Formes des matrices :
T1_dropna       : (6, 5)
T2_filter       : (5, 3)
T3_oversampling : (3, 4)


,nom,shape,nnz,taille_sparse_octets,taille_dense_theorique_octets,dense_vs_sparse
0,T_global_pipeline_3_operations,"(6, 4)",4,48,24,0.5


Provenance sortie finale -> entrée initiale
Ligne finale 0 <- ligne(s) initiale(s) 1
Ligne finale 1 <- ligne(s) initiale(s) 3
Ligne finale 2 <- ligne(s) initiale(s) 5
Ligne finale 3 <- ligne(s) initiale(s) 1

Provenance entrée initiale -> sortie finale
Ligne initiale 0 -> aucune ligne finale (exclue par le filtre)
Ligne initiale 1 -> ligne(s) finale(s) 0
Ligne initiale 2 -> aucune ligne finale (exclue par le filtre)
Ligne initiale 3 -> ligne(s) finale(s) 1
Ligne initiale 4 -> aucune ligne finale (exclue par le filtre)
Ligne initiale 5 -> ligne(s) finale(s) 2


Dans cet exemple, la matrice globale `T_global` permet de relier directement les lignes du dataset initial aux lignes du dataset final, même si plusieurs opérations ont été appliquées entre les deux.

La composition utilisée est :

`T_global = T1 @ T2 @ T3`

Cette écriture respecte l’orientation choisie dans ce notebook : chaque matrice de provenance a pour forme `lignes d'entrée × lignes de sortie`.

Ainsi, les lignes de `T_global` correspondent aux lignes du dataset initial, tandis que ses colonnes correspondent aux lignes du dataset final. On peut donc interroger la provenance dans les deux sens :

- sortie finale vers entrée initiale avec `output_to_inputs` ;
- entrée initiale vers sortie finale avec `input_to_outputs`.

Cet exemple répond à la deuxième tâche du projet, car il montre comment interroger la provenance non seulement pour une opération isolée, mais aussi pour une succession de plusieurs opérations dans un pipeline de préparation de données.

## 9 - Performance et benchmarks

Un autre objectif important du projet est d’évaluer l’efficacité de la méthode proposée.

Pour cela, nous utilisons des matrices creuses, notamment aux formats COO, CSR et CSC, afin de réduire l’espace mémoire nécessaire au stockage des tenseurs de provenance.

Le projet inclut également des benchmarks permettant de mesurer :

- le temps de construction des matrices de provenance ;
- le temps d’exécution des requêtes `output_to_inputs` et `input_to_outputs` ;
- la mémoire utilisée par une matrice creuse ;
- la mémoire théorique qu’aurait utilisée une matrice dense équivalente ;
- le gain mémoire obtenu grâce au stockage creux.


### 9.1 Complexité théorique


Opération filter — O(n_in)
- Construction : O(n_in) pour évaluer la condition + O(n_out) pour le tenseur COO
- Mémoire : O(n_out) en sparse (vs O(n_in × n_out) en dense)
- Requête : O(1) pour récupérer une ligne (une seule valeur non-nulle par colonne)

Opération merge — O(m + n + k)
- Construction : O(m + n) pour les colonnes sentinelles + O(k) pour les tenseurs CSR
- Mémoire : O(k) par tenseur (gain maximal si peu de matchs)
- Cas dégénéré : many-to-many peut donner k = m × n
- Requête : O(1) simple, O(k) inverse

Opération dropna — O(n_in × c)
- Construction : O(n_in × c) pour inspecter les NaN + O(n_out) pour le tenseur
- Mémoire : O(n_out) en sparse
- Requête : O(1)

Opération concat — O(m + n)
- Construction : O(m + n) via arithmétique d'indices directe (aucune inspection)
- Mémoire : O(m + n)
- Requête : O(1) (une seule valeur non-nulle par colonne)

In [301]:
def benchmark_filter(n=50_000):
    df = pd.DataFrame({"x": np.arange(n), "y": np.random.randn(n)})
    start = time.perf_counter()
    out, T = row_filter(df, "x % 2 == 0")
    elapsed = time.perf_counter() - start
    return {
        "operation": "filter",
        "n_input": len(df),
        "n_output": len(out),
        "temps_secondes": round(elapsed, 6),
        "nnz": T.nnz,
        "sparse_octets": tensor_nbytes(T),
        "dense_octets_theorique": dense_theoretical_nbytes(T.shape),
        "dense_vs_sparse": round(dense_theoretical_nbytes(T.shape) / max(tensor_nbytes(T), 1), 2),
    }


def benchmark_concat(n=25_000):
    df1 = pd.DataFrame({"x": np.arange(n)})
    df2 = pd.DataFrame({"x": np.arange(n, 2*n)})
    start = time.perf_counter()
    out, tensors = row_concat([df1, df2])
    elapsed = time.perf_counter() - start
    sparse_bytes = sum(tensor_nbytes(t) for t in tensors)
    dense_bytes = sum(dense_theoretical_nbytes(t.shape) for t in tensors)
    return {
        "operation": "concat",
        "n_input": 2*n,
        "n_output": len(out),
        "temps_secondes": round(elapsed, 6),
        "nnz": sum(t.nnz for t in tensors),
        "sparse_octets": sparse_bytes,
        "dense_octets_theorique": dense_bytes,
        "dense_vs_sparse": round(dense_bytes / max(sparse_bytes, 1), 2),
    }

def benchmark_merge(n=25_000):
    df1 = pd.DataFrame({"key": np.arange(n), "x": np.random.randn(n)})
    df2 = pd.DataFrame({"key": np.arange(n), "y": np.random.randn(n)})
    
    start = time.perf_counter()
    out, tensors = row_merge(df1, df2, on="key")
    elapsed = time.perf_counter() - start
    
    sparse_bytes = sum(tensor_nbytes(t) for t in tensors)
    dense_bytes = sum(dense_theoretical_nbytes(t.shape) for t in tensors)
    
    return {
        "operation": "merge",
        "n_input": 2*n,
        "n_output": len(out),
        "temps_secondes": round(elapsed, 6),
        "nnz": sum(t.nnz for t in tensors),
        "sparse_octets": sparse_bytes,
        "dense_octets_theorique": dense_bytes,
        "dense_vs_sparse": round(dense_bytes / max(sparse_bytes, 1), 2),
    }

def benchmark_dropna(n=50_000):
    df = pd.DataFrame({
        "x": np.concatenate([np.arange(n//2, dtype=float), np.full(n//2, np.nan)]),
        "y": np.random.randn(n)
})
    start = time.perf_counter()
    out, T = row_dropna(df)
    elapsed = time.perf_counter() - start
    return {
        "operation": "dropna",
        "n_input": len(df),
        "n_output": len(out),
        "temps_secondes": round(elapsed, 6),
        "nnz": T.nnz,
        "sparse_octets": tensor_nbytes(T),
        "dense_octets_theorique": dense_theoretical_nbytes(T.shape),
        "dense_vs_sparse": round(dense_theoretical_nbytes(T.shape) / max(tensor_nbytes(T), 1), 2),
    }

perf_df = pd.DataFrame([benchmark_filter(), benchmark_concat(), benchmark_merge(), benchmark_dropna()])
display(perf_df)


,operation,n_input,n_output,temps_secondes,nnz,sparse_octets,dense_octets_theorique,dense_vs_sparse
0,filter,50000,25000,0.001419,25000,325004,1250000000,3846.11
1,concat,50000,50000,0.000763,50000,450008,2500000000,5555.46
2,merge,50000,25000,0.002388,50000,450008,1250000000,2777.73
3,dropna,50000,25000,0.000695,25000,325004,1250000000,3846.11


### 9.2 Fonction de benchmark des requêtes de provenance

Afin de mesurer la performance des deux requêtes principales de provenance :

- `output_to_inputs` : retrouver les lignes d'entrée qui expliquent une ligne de sortie ;
- `input_to_outputs` : retrouver les lignes de sortie qui dépendent d'une ligne d'entrée.

Nous avons défini la fonction `benchmark_provenance_queries` qui exécute ces requêtes sur une matrice de provenance donnée, en mesurant le temps d'exécution et la mémoire utilisée.

In [302]:

def benchmark_provenance_queries(T, n_queries=1000, random_state=0):
    """
    Mesure le temps moyen des requêtes de provenance :
    - output_to_inputs : sortie -> entrée
    - input_to_outputs : entrée -> sortie

    Dans ce notebook, les tenseurs sont orientés ainsi :
    lignes d'entrée x lignes de sortie.
    """

    random.seed(random_state)

    n_inputs, n_outputs = T.shape

    output_indices = [
        random.randint(0, n_outputs - 1)
        for _ in range(min(n_queries, n_outputs))
    ]

    input_indices = [
        random.randint(0, n_inputs - 1)
        for _ in range(min(n_queries, n_inputs))
    ]

    # Temps des requêtes sortie -> entrée
    start = time.perf_counter()
    for out_row in output_indices:
        _ = output_to_inputs(T, out_row)
    output_to_input_time = time.perf_counter() - start

    # Temps des requêtes entrée -> sortie
    start = time.perf_counter()
    for in_row in input_indices:
        _ = input_to_outputs(T, in_row)
    input_to_output_time = time.perf_counter() - start

    results = {
        "tensor_shape": T.shape,
        "nnz": T.nnz,
        "density": T.nnz / (T.shape[0] * T.shape[1]),
        "n_output_to_inputs_queries": len(output_indices),
        "total_output_to_inputs_time_sec": output_to_input_time,
        "avg_output_to_inputs_time_ms": (output_to_input_time / len(output_indices)) * 1000,
        "n_input_to_outputs_queries": len(input_indices),
        "total_input_to_outputs_time_sec": input_to_output_time,
        "avg_input_to_outputs_time_ms": (input_to_output_time / len(input_indices)) * 1000,
    }

    return results

Pour compléter l'évaluation, on crée un dataset synthétique de grande taille. L'objectif est de tester le comportement des requêtes de provenance lorsque le nombre de lignes augmente.

On applique un filtre simple, puis on mesure le temps des requêtes de provenance sur le tenseur obtenu.

In [303]:
# Création d'un dataset synthétique plus grand

n = 100_000

large_df = pd.DataFrame({
    "id": np.arange(n),
    "score": np.random.randint(0, 100, size=n),
    "label": np.random.randint(0, 2, size=n)
})

large_filtered_df, T_large_filter = row_filter(
    large_df,
    "score >= 50"
)

print("Dimensions dataset initial :", large_df.shape)
print("Dimensions dataset filtré :", large_filtered_df.shape)

display(mapping_summary(T_large_filter, "T_large_filter"))

Dimensions dataset initial : (100000, 3)
Dimensions dataset filtré : (50089, 3)


,nom,shape,nnz,taille_sparse_octets,taille_dense_theorique_octets,dense_vs_sparse
0,T_large_filter,"(100000, 50089)",50089,650449,5008900000,7700.68


In [304]:
# Benchmark des requêtes de provenance sur le dataset synthétique

large_query_benchmark = benchmark_provenance_queries(
    T_large_filter,
    n_queries=10000,
    random_state=0
)

large_query_benchmark_df = pd.DataFrame([large_query_benchmark])
display(large_query_benchmark_df)

,tensor_shape,nnz,density,n_output_to_inputs_queries,total_output_to_inputs_time_sec,avg_output_to_inputs_time_ms,n_input_to_outputs_queries,total_input_to_outputs_time_sec,avg_input_to_outputs_time_ms
0,"(100000, 50089)",50089,0.00001,10000,3.015035,0.301503,10000,0.219301,0.02193


Le benchmark synthétique permet d'observer le comportement du système sur un nombre de lignes plus important.

Avec 100 000 lignes en entrée et environ 50 000 lignes conservées après filtrage, la matrice dense théorique serait très volumineuse. En revanche, la matrice creuse ne stocke que les relations réellement présentes, c'est-à-dire une relation par ligne conservée.

Le gain mémoire observé est donc très important, ce qui confirme l'intérêt du stockage sparse pour la provenance.

Les temps de requête restent raisonnables, même si `output_to_inputs` est plus lent que `input_to_outputs`. Cette différence s'explique par le format `csr_matrix`, qui est plus efficace pour accéder aux lignes que pour accéder aux colonnes.

Cette expérimentation complète l'évaluation réalisée sur Compass : Compass fournit un cas réel, tandis que le dataset synthétique permet de tester le passage à l'échelle.

### 9.4 Benchmark de la composition matricielle

Dans un pipeline composé de plusieurs opérations successives, la provenance globale peut être obtenue par multiplication de matrices creuses.

Cette section mesure le temps nécessaire pour calculer une provenance globale de type :

`T_global = T1 @ T2 @ T3`

L'objectif est d'évaluer non seulement le coût des requêtes de provenance, mais aussi le coût de construction d'un tenseur global reliant directement le dataset initial au dataset final.

In [305]:
def benchmark_matrix_composition(n=100_000, random_state=0):
    """
    Mesure le temps de composition de plusieurs matrices de provenance
    sur un pipeline synthétique composé de trois opérations :

    1. dropna simulé ;
    2. filtre sur score ;
    3. oversampling.

    La provenance globale est calculée avec :
    T_global = T1 @ T2 @ T3
    """

    np.random.seed(random_state)

    df_initial = pd.DataFrame({
        "id": np.arange(n),
        "score": np.random.randint(0, 100, size=n),
        "label": np.random.randint(0, 2, size=n)
    })

    # On introduit quelques valeurs manquantes pour simuler un dropna
    missing_indices = np.random.choice(
        df_initial.index,
        size=int(n * 0.05),
        replace=False
    )
    df_initial.loc[missing_indices, "score"] = np.nan

    # Étape 1 : dropna
    df_step1, T1 = row_dropna(df_initial)

    # Étape 2 : filtre
    df_step2, T2 = row_filter(df_step1, "score >= 50")

    # Étape 3 : oversampling
    df_final, T3 = row_random_oversample(
        df_step2,
        target_col="label",
        random_state=random_state
    )

    # Mesure du temps de composition
    start = time.perf_counter()

    T_global = T1 @ T2 @ T3
    T_global = T_global.astype(bool).astype(np.int8).tocsr()

    composition_time = time.perf_counter() - start

    result = {
        "n_initial_rows": df_initial.shape[0],
        "n_after_dropna": df_step1.shape[0],
        "n_after_filter": df_step2.shape[0],
        "n_final_rows": df_final.shape[0],
        "T1_shape": T1.shape,
        "T2_shape": T2.shape,
        "T3_shape": T3.shape,
        "T_global_shape": T_global.shape,
        "T_global_nnz": T_global.nnz,
        "T_global_density": T_global.nnz / (T_global.shape[0] * T_global.shape[1]),
        "composition_time_sec": composition_time,
    }

    return result, T_global

In [306]:
results = pd.DataFrame()

for size in [10_000, 100_000, 1_000_000]:
    composition_result, T_global_large = benchmark_matrix_composition(
        n=size,
        random_state=0
    )
    results = pd.concat([results, pd.DataFrame([composition_result])], ignore_index=True)

display(results)

,n_initial_rows,n_after_dropna,n_after_filter,n_final_rows,T1_shape,T2_shape,T3_shape,T_global_shape,T_global_nnz,T_global_density,composition_time_sec
0,10000,9500,4652,4754,"(10000, 9500)","(9500, 4652)","(4652, 4754)","(10000, 4754)",4754,0.000100,0.000354
1,100000,95000,47552,47614,"(100000, 95000)","(95000, 47552)","(47552, 47614)","(100000, 47614)",47614,0.000010,0.002972
2,1000000,950000,474993,475102,"(1000000, 950000)","(950000, 474993)","(474993, 475102)","(1000000, 475102)",475102,0.000001,0.026743


Ce benchmark mesure le coût de construction d'une provenance globale à partir de plusieurs tenseurs intermédiaires.

La composition `T_global = T1 @ T2 @ T3` permet de relier directement les lignes du dataset final aux lignes du dataset initial, sans devoir parcourir manuellement chaque étape du pipeline.

Le résultat montre que la composition reste réalisable grâce au caractère creux des matrices. Comme chaque opération conserve un nombre limité de relations effectives, le nombre de valeurs non nulles reste faible par rapport à la taille dense théorique.

Cette mesure complète les benchmarks précédents : on évalue désormais à la fois le coût de stockage, le temps des requêtes de provenance et le temps de calcul de la provenance globale.

## 10 - Evaluation sur les différents pipelines de nettoyage

Les datasets ne sont pas toujours dans le même dossier selon l'environnement utilisé.  
La fonction `find_dataset()` cherche automatiquement les fichiers dans plusieurs emplacements possibles.

La fonction `load_dataset()` charge ensuite le fichier trouvé. Si le fichier est absent, elle lève une erreur explicite afin d'éviter de poursuivre l'exécution avec un dataset manquant.


In [307]:
def find_dataset(filename):
    candidates = [
        filename,
        os.path.join(".", filename),
        os.path.join("Datasets", filename),
        os.path.join("UseCases", "Datasets", filename),
        os.path.join("consigne", "UseCases", "Datasets", filename),
        os.path.join(
            "/content",
            "drive",
            "MyDrive",
            "ExecMaster-MiniProjet_Data Quality",
            "UseCases",
            "Datasets",
            filename,
        ),
    ]

    for path in candidates:
        if os.path.exists(path):
            return path

    return None


def load_dataset(filename, **kwargs):
    path = find_dataset(filename)

    if path is None:
        raise FileNotFoundError(
            f"Dataset {filename!r} introuvable. "
            "Vérifiez que le fichier est placé dans le dossier UseCases/Datasets."
        )

    return pd.read_csv(path, **kwargs)

### 10.1. Pipeline Compass



#### 10.1.1. Application au pipeline

Dans le script Compass, les opérations principales sont : sélection de colonnes, `dropna`, binarisation de `race`, renommage et inversion de `two_year_recid` en `label`, création de `jailtime`, suppression des dates de prison, puis binarisation de `c_charge_degree`.


In [308]:
start_time = time.perf_counter()

df_original = load_dataset("compas.csv", header=0)

display(df_original.head())
print("Dimensions du dataset Compass brut :", df_original.shape)

row_prov_dropna = RowProv(row_dropna, operation="dropna")
col_prov = ColumnProv(df_original)

# OPÉRATION 0 : Sélection des colonnes pertinentes
selected_cols = [
    "age",
    "c_charge_degree",
    "race",
    "sex",
    "priors_count",
    "days_b_screening_arrest",
    "two_year_recid",
    "c_jail_in",
    "c_jail_out",
]

df = df_original[selected_cols].copy()
col_prov.select(selected_cols)

# OPÉRATION 1 : Suppression des lignes contenant des valeurs manquantes
df, T_dropna = row_prov_dropna(df)

# OPÉRATION 2 : Binarisation de la variable race
df["race"] = df["race"].apply(lambda r: 1 if r == "Caucasian" else 0)
col_prov.transform_column("race")

# OPÉRATION 3 : Renommage de two_year_recid en label et inversion des valeurs
df = df.rename({"two_year_recid": "label"}, axis=1)
col_prov.rename_column("two_year_recid", "label")

df["label"] = df["label"].apply(lambda l: 0 if l == 1 else 1)
col_prov.transform_column("label")

# OPÉRATION 4 : Calcul de la durée d'incarcération en jours
df["jailtime"] = (
    pd.to_datetime(df["c_jail_out"])
    - pd.to_datetime(df["c_jail_in"])
).dt.days

col_prov.add_column("jailtime", from_cols=["c_jail_in", "c_jail_out"])

# OPÉRATION 5 : Suppression des colonnes de dates d'entrée et de sortie de prison
df = df.drop(columns=["c_jail_in", "c_jail_out"])
col_prov.drop_columns(["c_jail_in", "c_jail_out"])

# OPÉRATION 6 : Binarisation de la variable c_charge_degree
df["c_charge_degree"] = df["c_charge_degree"].apply(lambda s: 1 if s == "F" else 0)
col_prov.transform_column("c_charge_degree")

# Conservation du dataset Compass nettoyé pour les analyses complémentaires de provenance
compas_clean = df.copy()
T_compas_dropna = T_dropna

compas_processing_time = time.perf_counter() - start_time

display(compas_clean.head())
print("Dimensions du dataset Compass nettoyé :", compas_clean.shape)
print("Temps de traitement Compass :", round(compas_processing_time, 4), "secondes")

print("Résumé du tenseur de provenance des lignes après dropna :")
display(mapping_summary(T_compas_dropna, "T_compas_dropna"))

print("Provenance des colonnes :")
col_prov.print()

for out_row in range(min(5, len(compas_clean))):
    print(
        f"Ligne Compass finale {out_row} <- ligne d'origine "
        f"{output_to_inputs(T_compas_dropna, out_row)}"
    )

,id,name,first,last,compas_screening_date,sex,dob,age,age_cat,race,...,v_decile_score,v_score_text,v_screening_date,in_custody,out_custody,priors_count.1,start,end,event,two_year_recid
0,1,miguel hernandez,miguel,hernandez,2013-08-14,Male,1947-04-18,69,Greater than 45,Other,...,1,Low,2013-08-14,2014-07-07,2014-07-14,0,0,327,0,0
1,3,kevon dixon,kevon,dixon,2013-01-27,Male,1982-01-22,34,25 - 45,African-American,...,1,Low,2013-01-27,2013-01-26,2013-02-05,0,9,159,1,1
2,4,ed philo,ed,philo,2013-04-14,Male,1991-05-14,24,Less than 25,African-American,...,3,Low,2013-04-14,2013-06-16,2013-06-16,4,0,63,0,1
3,5,marcu brown,marcu,brown,2013-01-13,Male,1993-01-21,23,Less than 25,African-American,...,6,Medium,2013-01-13,NaN,NaN,1,0,1174,0,0
4,6,bouthy pierrelouis,bouthy,pierrelouis,2013-03-26,Male,1973-01-22,43,25 - 45,Other,...,1,Low,2013-03-26,NaN,NaN,2,0,1102,0,0


Dimensions du dataset Compass brut : (7214, 53)


,age,c_charge_degree,race,sex,priors_count,days_b_screening_arrest,label,jailtime
0,69,1,0,Male,0,-1.0,1,0
1,34,1,0,Male,0,-1.0,0,10
2,24,1,0,Male,4,-1.0,0,1
3,44,0,0,Male,0,0.0,1,1
4,41,1,1,Male,14,-1.0,0,6


Dimensions du dataset Compass nettoyé : (6907, 8)
Temps de traitement Compass : 0.038 secondes
Résumé du tenseur de provenance des lignes après dropna :


,nom,shape,nnz,taille_sparse_octets,taille_dense_theorique_octets,dense_vs_sparse
0,T_compas_dropna,"(7214, 6907)",6907,63395,49827098,785.98


Provenance des colonnes :


,Colonne finale,Colonne(s) d'origine
0,age,age
1,c_charge_degree,c_charge_degree
2,race,race
3,sex,sex
4,priors_count,priors_count
5,days_b_screening_arrest,days_b_screening_arrest
6,label,label
7,jailtime,"c_jail_in, c_jail_out"


Ligne Compass finale 0 <- ligne d'origine [np.int32(0)]
Ligne Compass finale 1 <- ligne d'origine [np.int32(1)]
Ligne Compass finale 2 <- ligne d'origine [np.int32(2)]
Ligne Compass finale 3 <- ligne d'origine [np.int32(5)]
Ligne Compass finale 4 <- ligne d'origine [np.int32(6)]


#### 10.1.2. Filtrage avec provenance sur le dataset Compass

Pour renforcer l'analyse de la provenance des lignes sur un dataset réel, on applique ici une opération de filtrage explicite sur le dataset Compass nettoyé.

L'opération choisie consiste à conserver uniquement les individus ayant au moins un antécédent judiciaire (`priors_count >= 1`). Cette opération correspond à une réduction horizontale du dataset, car elle réduit le nombre de lignes tout en conservant les mêmes colonnes.

Le tenseur de provenance permet ensuite de savoir quelles lignes du dataset Compass nettoyé ont été conservées dans le dataset filtré.

In [309]:
# Filtrage explicite sur Compass avec capture de provenance
# Condition : conserver les individus ayant au moins un antécédent judiciaire

compas_filtered, T_compas_filter = row_filter(
    compas_clean,
    "priors_count >= 1"
)

print("Dimensions du dataset Compass nettoyé :")
print(compas_clean.shape)

print("\nDimensions du dataset Compass après filtrage priors_count >= 1 :")
print(compas_filtered.shape)

display(compas_clean.head())
display(compas_filtered.head())

display(mapping_summary(T_compas_filter, "T_compas_filter_priors_count"))



Dimensions du dataset Compass nettoyé :
(6907, 8)

Dimensions du dataset Compass après filtrage priors_count >= 1 :
(4806, 8)


,age,c_charge_degree,race,sex,priors_count,days_b_screening_arrest,label,jailtime
0,69,1,0,Male,0,-1.0,1,0
1,34,1,0,Male,0,-1.0,0,10
2,24,1,0,Male,4,-1.0,0,1
3,44,0,0,Male,0,0.0,1,1
4,41,1,1,Male,14,-1.0,0,6


,age,c_charge_degree,race,sex,priors_count,days_b_screening_arrest,label,jailtime
0,24,1,0,Male,4,-1.0,0,1
1,41,1,1,Male,14,-1.0,0,6
2,43,1,0,Male,3,-1.0,1,0
3,21,1,1,Male,1,428.0,0,0
4,23,0,0,Male,3,0.0,0,4


,nom,shape,nnz,taille_sparse_octets,taille_dense_theorique_octets,dense_vs_sparse
0,T_compas_filter_priors_count,"(6907, 4806)",4806,51662,33195042,642.54


In [310]:
print("Exemples : provenance sortie -> entrée")
for out_row in range(min(10, T_compas_filter.shape[1])):
    input_rows = output_to_inputs(T_compas_filter, out_row)
    print(f"Ligne filtrée {out_row} <- ligne(s) Compass nettoyé {input_rows[0]}")

print("\nExemples : provenance entrée -> sortie")
for in_row in range(min(10, T_compas_filter.shape[0])):
    output_rows = input_to_outputs(T_compas_filter, in_row)
    if len(output_rows) > 0:
        print(f"Ligne Compass nettoyé {in_row} -> ligne(s) filtrée(s) {output_rows[0]}")
    else:
        print(f"Ligne Compass nettoyé {in_row} -> aucune ligne filtrée (exclue par le filtre)")

Exemples : provenance sortie -> entrée
Ligne filtrée 0 <- ligne(s) Compass nettoyé 2
Ligne filtrée 1 <- ligne(s) Compass nettoyé 4
Ligne filtrée 2 <- ligne(s) Compass nettoyé 5
Ligne filtrée 3 <- ligne(s) Compass nettoyé 7
Ligne filtrée 4 <- ligne(s) Compass nettoyé 9
Ligne filtrée 5 <- ligne(s) Compass nettoyé 12
Ligne filtrée 6 <- ligne(s) Compass nettoyé 13
Ligne filtrée 7 <- ligne(s) Compass nettoyé 15
Ligne filtrée 8 <- ligne(s) Compass nettoyé 16
Ligne filtrée 9 <- ligne(s) Compass nettoyé 17

Exemples : provenance entrée -> sortie
Ligne Compass nettoyé 0 -> aucune ligne filtrée (exclue par le filtre)
Ligne Compass nettoyé 1 -> aucune ligne filtrée (exclue par le filtre)
Ligne Compass nettoyé 2 -> ligne(s) filtrée(s) 0
Ligne Compass nettoyé 3 -> aucune ligne filtrée (exclue par le filtre)
Ligne Compass nettoyé 4 -> ligne(s) filtrée(s) 1
Ligne Compass nettoyé 5 -> ligne(s) filtrée(s) 2
Ligne Compass nettoyé 6 -> aucune ligne filtrée (exclue par le filtre)
Ligne Compass nettoyé 7 -

### 10.1.3. Benchmark des requêtes de provenance sur Compass

Le sujet demande d'évaluer non seulement l'espace mémoire nécessaire pour stocker les tenseurs de provenance, mais aussi le temps nécessaire pour interroger cette provenance.

Dans cette section, on mesure donc le temps d'exécution des deux requêtes principales sur le tenseur `T_compas_filter`, obtenu à partir du filtrage réel du dataset Compass.

In [311]:
# Benchmark des requêtes de provenance sur Compass

compas_query_benchmark = benchmark_provenance_queries(
    T_compas_filter,
    n_queries=1000,
    random_state=0
)

compas_query_benchmark_df = pd.DataFrame([compas_query_benchmark])
display(compas_query_benchmark_df)


,tensor_shape,nnz,density,n_output_to_inputs_queries,total_output_to_inputs_time_sec,avg_output_to_inputs_time_ms,n_input_to_outputs_queries,total_input_to_outputs_time_sec,avg_input_to_outputs_time_ms
0,"(6907, 4806)",4806,0.000145,1000,0.048057,0.048057,1000,0.027637,0.027637


Les résultats montrent que les requêtes de provenance restent raisonnablement rapides sur le dataset Compass.

Le tenseur utilisé est une matrice creuse binaire. Il évite de stocker toutes les combinaisons possibles entre lignes d'entrée et lignes de sortie : seules les relations réellement existantes sont conservées.

La densité du tenseur est très faible, ce qui confirme l'intérêt des matrices creuses pour représenter la provenance dans des pipelines de préparation de données.

On observe cependant une différence entre les deux types de requêtes. La requête `input_to_outputs` est plus rapide, car la matrice est stockée au format `csr_matrix`, qui est optimisé pour l'accès aux lignes. À l'inverse, `output_to_inputs` revient à interroger une colonne de la matrice, ce qui est généralement moins efficace avec ce format.

Ces mesures répondent à l'objectif du projet, qui demande d'évaluer le temps nécessaire au traitement des requêtes de provenance.

### 10.2. Application au pipeline German

Dans le script German, il n'y a pas de suppression de lignes. L'essentiel de la provenance concerne donc les colonnes : remplacement de valeurs, création de `status` et `gender` depuis `personal_status`, suppression de `personal_status`, puis one-hot encoding des variables catégorielles.


In [312]:
start_time = time.perf_counter()

df_german_original = load_dataset("german.csv", header=0)

display(df_german_original.head())
print("Dimensions du dataset German brut :", df_german_original.shape)

df = df_german_original.copy()
col_prov = ColumnProv(df)

# OPÉRATION 0 : Remplacement des codes par des libellés interprétables
df = df.replace({'checking': {'A11': 'check_low', 'A12': 'check_mid', 'A13': 'check_high', 'A14': 'check_none'},
                  'credit_history': {'A30': 'debt_none', 'A31': 'debt_noneBank', 'A32': 'debt_onSchedule', 'A33': 'debt_delay', 'A34': 'debt_critical'},
                  'purpose': {'A40': 'pur_newCar', 'A41': 'pur_usedCar', 'A42': 'pur_furniture', 'A43': 'pur_tv',
                              'A44': 'pur_appliance', 'A45': 'pur_repairs', 'A46': 'pur_education', 'A47': 'pur_vacation',
                              'A48': 'pur_retraining', 'A49': 'pur_business', 'A410': 'pur_other'},
                  'savings': {'A61': 'sav_small', 'A62': 'sav_medium', 'A63': 'sav_large', 'A64': 'sav_xlarge', 'A65': 'sav_none'},
                  'employment': {'A71': 'emp_unemployed', 'A72': 'emp_lessOne', 'A73': 'emp_lessFour', 'A74': 'emp_lessSeven', 'A75': 'emp_moreSeven'},
                  'other_debtors': {'A101': 'debtor_none', 'A102': 'debtor_coApp', 'A103': 'debtor_guarantor'},
                  'property': {'A121': 'prop_realEstate', 'A122': 'prop_agreement', 'A123': 'prop_car', 'A124': 'prop_none'},
                  'other_inst': {'A141': 'oi_bank', 'A142': 'oi_stores', 'A143': 'oi_none'},
                  'housing': {'A151': 'hous_rent', 'A152': 'hous_own', 'A153': 'hous_free'},
                  'job': {'A171': 'job_unskilledNR', 'A172': 'job_unskilledR', 'A173': 'job_skilled', 'A174': 'job_highSkill'},
                  'phone': {'A191': 0, 'A192': 1},
                  'foreigner': {'A201': 1, 'A202': 0},
                  'label': {2: 0}})

# OPÉRATION 1 : Création des variables status et gender à partir de personal_status
df['status'] = df['personal_status'].map({'A91': 'divorced', 'A92': 'divorced', 'A93': 'single', 'A95': 'single'}).fillna('married')
df['gender'] = df['personal_status'].map({'A92': 0, 'A95': 0}).fillna(1)
col_prov.add_column('status', from_cols=['personal_status'])
col_prov.add_column('gender', from_cols=['personal_status'])

# OPÉRATION 2 : Suppression de la colonne personal_status
df.drop(columns=['personal_status'], inplace=True)
col_prov.drop_columns(['personal_status'])

# OPÉRATION 3 : Encodage one-hot des variables catégorielles
categorical_cols = ['checking', 'credit_history', 'purpose', 'savings', 'employment', 'other_debtors',
                    'property', 'other_inst', 'housing', 'job', 'status']
df_before_dummies = df.copy()
col_prov.to_dummies(categorical_cols, df_before_dummies=df_before_dummies)
df = pd.get_dummies(df, columns=categorical_cols)

# Conservation du dataset German traité pour la comparaison finale
german_clean = df.copy()

german_processing_time = time.perf_counter() - start_time

display(german_clean.head())
print("Dimensions du dataset German traité :", german_clean.shape)
print("Temps de traitement German :", round(german_processing_time, 4), "secondes")

print("Provenance des colonnes German :")
col_prov.print()


,checking,duration,credit_history,purpose,credit_amount,savings,employment,inst_rate,personal_status,other_debtors,...,property,age,other_inst,housing,num_credits,job,dependants,phone,foreigner,label
0,A11,6,A34,A43,1169,A65,A75,4,A93,A101,...,A121,67,A143,A152,2,A173,1,A192,A201,1
1,A12,48,A32,A43,5951,A61,A73,2,A92,A101,...,A121,22,A143,A152,1,A173,1,A191,A201,2
2,A14,12,A34,A46,2096,A61,A74,2,A93,A101,...,A121,49,A143,A152,1,A172,2,A191,A201,1
3,A11,42,A32,A42,7882,A61,A74,2,A93,A103,...,A122,45,A143,A153,1,A173,2,A191,A201,1
4,A11,24,A33,A40,4870,A61,A73,3,A93,A101,...,A124,53,A143,A153,2,A173,2,A191,A201,2


Dimensions du dataset German brut : (1000, 21)


,duration,credit_amount,inst_rate,residence_time,age,num_credits,dependants,phone,foreigner,label,...,housing_hous_free,housing_hous_own,housing_hous_rent,job_job_highSkill,job_job_skilled,job_job_unskilledNR,job_job_unskilledR,status_divorced,status_married,status_single
0,6,1169,4,4,67,2,1,1,1,1,...,False,True,False,False,True,False,False,False,False,True
1,48,5951,2,2,22,1,1,0,1,0,...,False,True,False,False,True,False,False,True,False,False
2,12,2096,2,3,49,1,2,0,1,1,...,False,True,False,False,False,False,True,False,False,True
3,42,7882,2,4,45,1,2,0,1,1,...,True,False,False,False,True,False,False,False,False,True
4,24,4870,3,4,53,2,2,0,1,0,...,True,False,False,False,True,False,False,False,False,True


Dimensions du dataset German traité : (1000, 60)
Temps de traitement German : 0.015 secondes
Provenance des colonnes German :


,Colonne finale,Colonne(s) d'origine
0,duration,duration
1,credit_amount,credit_amount
2,inst_rate,inst_rate
3,residence_time,residence_time
4,age,age
5,num_credits,num_credits
6,dependants,dependants
7,phone,phone
8,foreigner,foreigner
9,label,label


### 10.3. Application au pipeline Census

Dans le script Census, les colonnes sont d'abord renommées, les espaces sont nettoyés, les `?` sont remplacés par `NaN`, plusieurs variables catégorielles sont encodées en one-hot, `sex` et `label` sont binarisées, puis `fnlwgt` est supprimée.


In [313]:
start_time = time.perf_counter()

df_census_original = load_dataset("census.csv", header=0)

display(df_census_original.head())
print("Dimensions du dataset Census brut :", df_census_original.shape)

df = df_census_original.copy()


names = ['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status', 'occupation',
          'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week',
          'native-country', 'label']
df.columns = names
col_prov = ColumnProv(df)

# OPÉRATION 0 : Nettoyage des espaces dans les colonnes catégorielles
strip_cols = ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country', 'label']
for c in strip_cols:
    df[c] = df[c].map(str.strip)
    col_prov.transform_column(c)

# OPÉRATION 1 : Remplacement des valeurs ? par NaN
df = df.replace('?', np.nan)

# OPÉRATION 2 : Encodage one-hot des variables catégorielles
categorical_cols = ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'native-country']
df_before_dummies = df.copy()
col_prov.to_dummies(categorical_cols, df_before_dummies=df_before_dummies)
df = pd.get_dummies(df, columns=categorical_cols)

# OPÉRATION 3 : Binarisation des variables sex et label
df.sex = df.sex.replace({'Male': 1, 'Female': 0})
col_prov.transform_column('sex')
df.label = df.label.replace({'<=50K': 0, '>50K': 1})
col_prov.transform_column('label')

# OPÉRATION 4 : Suppression de la colonne fnlwgt
df = df.drop(['fnlwgt'], axis=1)
col_prov.drop_columns(['fnlwgt'])

# Conservation du dataset Census traité pour la comparaison finale
census_clean = df.copy()

census_processing_time = time.perf_counter() - start_time

display(census_clean.head())
print("Dimensions du dataset census traité :", census_clean.shape)
print("Temps de traitement census :", round(census_processing_time, 4), "secondes")

print("Provenance des colonnes census :")
col_prov.print()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


Dimensions du dataset Census brut : (32561, 15)


,age,education-num,sex,capital-gain,capital-loss,hours-per-week,label,workclass_Federal-gov,workclass_Local-gov,workclass_Never-worked,...,native-country_Portugal,native-country_Puerto-Rico,native-country_Scotland,native-country_South,native-country_Taiwan,native-country_Thailand,native-country_Trinadad&Tobago,native-country_United-States,native-country_Vietnam,native-country_Yugoslavia
0,39,13,1,2174,0,40,0,False,False,False,...,False,False,False,False,False,False,False,True,False,False
1,50,13,1,0,0,13,0,False,False,False,...,False,False,False,False,False,False,False,True,False,False
2,38,9,1,0,0,40,0,False,False,False,...,False,False,False,False,False,False,False,True,False,False
3,53,7,1,0,0,40,0,False,False,False,...,False,False,False,False,False,False,False,True,False,False
4,28,13,0,0,0,40,0,False,False,False,...,False,False,False,False,False,False,False,False,False,False


Dimensions du dataset census traité : (32561, 104)
Temps de traitement census : 0.0758 secondes
Provenance des colonnes census :


,Colonne finale,Colonne(s) d'origine
0,age,age
1,education-num,education-num
2,sex,sex
3,capital-gain,capital-gain
4,capital-loss,capital-loss
...,...,...
99,native-country_Thailand,native-country
100,native-country_Trinadad&Tobago,native-country
101,native-country_United-States,native-country
102,native-country_Vietnam,native-country


## 11 - Comparaison des traitements sur les datasets réels

In [314]:
def dataframe_memory_bytes(df):
    return df.memory_usage(deep=True).sum()


def dataset_summary(name, raw_df, processed_df, row_tensor=None, processing_time_sec=None):
    """
    Résume les dimensions, la mémoire, le temps de traitement
    et éventuellement la provenance des lignes pour un dataset réel.
    """
    result = {
        "dataset": name,
        "rows_before": raw_df.shape[0],
        "cols_before": raw_df.shape[1],
        "rows_after": processed_df.shape[0],
        "cols_after": processed_df.shape[1],
        "memory_before_bytes": dataframe_memory_bytes(raw_df),
        "memory_after_bytes": dataframe_memory_bytes(processed_df),
        "processing_time_sec": processing_time_sec,
        "row_provenance_available": row_tensor is not None,
    }

    if row_tensor is not None:
        result["tensor_shape"] = row_tensor.shape
        result["tensor_nnz"] = row_tensor.nnz
        result["tensor_density"] = row_tensor.nnz / (
            row_tensor.shape[0] * row_tensor.shape[1]
        )
    else:
        result["tensor_shape"] = None
        result["tensor_nnz"] = None
        result["tensor_density"] = None

    return result

In [315]:
# Construction du tableau comparatif des datasets réels

summaries = [
    dataset_summary(
        name="Compass",
        raw_df=df_original,
        processed_df=compas_filtered,
        row_tensor=T_compas_filter,
        processing_time_sec=compas_processing_time
    ),
    dataset_summary(
        name="German",
        raw_df=df_german_original,
        processed_df=german_clean,
        row_tensor=None,
        processing_time_sec=german_processing_time
    ),
    dataset_summary(
        name="Census",
        raw_df=df_census_original,
        processed_df=census_clean,
        row_tensor=None,
        processing_time_sec=census_processing_time
    )
]

real_datasets_summary_df = pd.DataFrame(summaries)

display(real_datasets_summary_df)

,dataset,rows_before,cols_before,rows_after,cols_after,memory_before_bytes,memory_after_bytes,processing_time_sec,row_provenance_available,tensor_shape,tensor_nnz,tensor_density
0,Compass,7214,53,4806,8,15222914,564026,0.038035,True,"(6907, 4806)",4806.0,0.000145
1,German,1000,21,1000,60,851144,193132,0.015026,False,None,NaN,NaN
2,Census,32561,15,32561,104,21141580,6805381,0.075775,False,None,NaN,NaN


Le tableau comparatif inclut désormais le temps de traitement de chaque pipeline réel.

Cette mesure permet de compléter l'évaluation demandée dans le sujet : on ne compare pas seulement l'espace mémoire occupé par les DataFrames et les tenseurs de provenance, mais aussi le temps nécessaire pour appliquer les traitements de préparation des données.

Compass inclut une réduction horizontale avec `dropna` puis un filtrage explicite `priors_count >= 1`, ce qui permet de construire une provenance des lignes. German et Census comportent surtout des transformations verticales, notamment des remplacements de valeurs, des suppressions de colonnes et du one-hot encoding.

Le temps de traitement permet donc de comparer le coût pratique des pipelines réels en plus de leurs effets sur les dimensions des datasets.

In [316]:
print("Validation finale du notebook")
print("Compass filtré :", compas_filtered.shape)
print("German traité :", german_clean.shape)
print("Census traité :", census_clean.shape)
print("Tenseur Compass :", T_compas_filter.shape, "nnz =", T_compas_filter.nnz)
print("Tableau comparatif :", real_datasets_summary_df.shape)

Validation finale du notebook
Compass filtré : (4806, 8)
German traité : (1000, 60)
Census traité : (32561, 104)
Tenseur Compass : (6907, 4806) nnz = 4806
Tableau comparatif : (3, 12)


## 12 - Optimisation des requêtes avec CSR et CSC

Les résultats précédents montrent que la requête `output_to_inputs` est généralement plus lente que `input_to_outputs`.

Cela s'explique par le format `csr_matrix`, qui est optimisé pour accéder aux lignes de la matrice. Or, dans ce notebook, les matrices de provenance sont orientées comme suit :

- les lignes correspondent aux lignes d'entrée ;
- les colonnes correspondent aux lignes de sortie.

Ainsi, `input_to_outputs` interroge une ligne de la matrice, ce qui est efficace en CSR. À l'inverse, `output_to_inputs` interroge une colonne, ce qui est moins efficace en CSR.

Pour optimiser les deux types de requêtes, on prépare donc deux formats de la même matrice :

- CSR pour `input_to_outputs` ;
- CSC pour `output_to_inputs`.

In [317]:
def prepare_query_matrices(T):
    """
    Prépare deux formats de la même matrice de provenance :
    - CSR pour les requêtes input_to_outputs ;
    - CSC pour les requêtes output_to_inputs.
    """
    return {
        "csr": T.tocsr(),
        "csc": T.tocsc()
    }


def output_to_inputs_fast(prepared_T, out_row):
    """
    Version optimisée de output_to_inputs.

    Elle utilise le format CSC, plus efficace pour accéder aux colonnes.
    """
    m = prepared_T["csc"]
    return list(m[:, out_row].nonzero()[0])


def input_to_outputs_fast(prepared_T, input_row):
    """
    Version optimisée de input_to_outputs.

    Elle utilise le format CSR, plus efficace pour accéder aux lignes.
    """
    m = prepared_T["csr"]
    return list(m[input_row, :].nonzero()[1])

In [318]:
# Préparation des deux formats de la matrice Compass
prepared_compas_T = prepare_query_matrices(T_compas_filter)

n_queries = 1000

output_indices = list(range(min(n_queries, T_compas_filter.shape[1])))
input_indices = list(range(min(n_queries, T_compas_filter.shape[0])))

# Benchmark output_to_inputs classique
start = time.perf_counter()
for out_row in output_indices:
    _ = output_to_inputs(T_compas_filter, out_row)
classic_output_time = time.perf_counter() - start

# Benchmark output_to_inputs optimisé
start = time.perf_counter()
for out_row in output_indices:
    _ = output_to_inputs_fast(prepared_compas_T, out_row)
fast_output_time = time.perf_counter() - start

# Benchmark input_to_outputs classique
start = time.perf_counter()
for in_row in input_indices:
    _ = input_to_outputs(T_compas_filter, in_row)
classic_input_time = time.perf_counter() - start

# Benchmark input_to_outputs optimisé
start = time.perf_counter()
for in_row in input_indices:
    _ = input_to_outputs_fast(prepared_compas_T, in_row)
fast_input_time = time.perf_counter() - start

optimization_results = pd.DataFrame([
    {
        "query": "output_to_inputs",
        "classic_time_sec": classic_output_time,
        "optimized_time_sec": fast_output_time,
        "gain": classic_output_time / fast_output_time
    },
    {
        "query": "input_to_outputs",
        "classic_time_sec": classic_input_time,
        "optimized_time_sec": fast_input_time,
        "gain": classic_input_time / fast_input_time
    }
])

display(optimization_results)

,query,classic_time_sec,optimized_time_sec,gain
0,output_to_inputs,0.035142,0.012226,2.874387
1,input_to_outputs,0.028288,0.028276,1.000420


Le tableau compare les requêtes classiques et les requêtes optimisées.

L'amélioration la plus importante est attendue sur `output_to_inputs`, car cette requête interroge les colonnes de la matrice. Le passage au format CSC permet donc d'accélérer cette opération.

Pour `input_to_outputs`, le gain peut être plus faible, car la matrice initiale est déjà au format CSR, qui est adapté à l'accès aux lignes.

Cette optimisation montre que le choix du format de matrice creuse dépend du sens des requêtes de provenance. Une même matrice peut donc être conservée sous deux formats complémentaires afin d'améliorer les performances de navigation dans la provenance.

## 13 - Conclusion

Nous avons développé une solution de capture de provenance basée sur des tenseurs binaires creux. La classe `RowProv` permet de capturer la provenance des opérations qui modifient le nombre ou l'origine des lignes : filtrage, `dropna`, `merge`, `concat` et oversampling. La classe `ColumnProv` complète cette approche en documentant la provenance des colonnes lors des transformations de schéma.

Les requêtes de provenance permettent de naviguer dans les deux sens : depuis une ligne de sortie vers ses lignes d'entrée, et depuis une ligne d'entrée vers les lignes de sortie qui en dépendent. Pour les pipelines successifs, la provenance peut être composée par multiplication matricielle sparse. Cette propriété est utile pour remonter jusqu'aux données initiales même après plusieurs étapes de traitement.

Le **stockage sparse est adapté** au projet car les tenseurs de provenance **contiennent très peu de valeurs** `1` par rapport au nombre total de positions possibles. Il permet donc de réduire fortement l'espace mémoire par rapport à une matrice dense, tout en conservant une interrogation efficace de la provenance.

L'application aux datasets réels montre que les opérations n'ont pas toutes le même impact sur la provenance. Compass permet d'observer une vraie provenance des lignes grâce aux opérations `dropna` et au filtrage explicite `priors_count >= 1`. German et Census illustrent davantage les transformations verticales, notamment les remplacements de valeurs, les suppressions de colonnes et le one-hot encoding.

Toutefois, les datasets German et Census ne comportent pas de réduction horizontale explicite dans les traitements appliqués : aucune opération de filtrage de lignes, de suppression de lignes ou de sur-échantillonnage n'est réalisée dans ces deux pipelines. C'est pourquoi aucun tenseur de provenance des lignes n'est construit pour ces datasets dans le tableau comparatif. Leur traitement relève principalement de la provenance des colonnes, avec des opérations comme le remplacement de valeurs, la suppression de colonnes et l'encodage one-hot.

Les benchmarks réalisés confirment l'intérêt du stockage sparse : **les matrices de provenance sont très peu denses** et permettent un **gain mémoire important** par rapport à une matrice dense théorique. **Les temps de requête restent raisonnables**, même sur un dataset synthétique plus grand, ce qui montre que l'approche peut passer à l'échelle pour des pipelines de préparation de données.

## 14 - Pour aller plus loin

Nous aurions pu étendre la consigne du projet pour capturer (au delà de la provenance des lignes et des colonnes) la provenance au niveau des cellules individuelles d'un DataFrame. Cela aurait permis de suivre précisément l'origine de chaque valeur dans le dataset final, même après des transformations complexes.